# Imports and Libraries

## Installs

Dataset and code for downloading found:
https://huggingface.co/docs/datasets/v1.13.2/load_hub.html

In [2]:
# Commented out to stop multiple runs when running entire notebook
#%pip install -U datasets

## Imports

`load_dataset` - For downloading sst2 dataset

`DatasetDict` - For creating project dataset splits

`pd` - For usage of DataFrame type

`AutoTokenizer` - For tokenisation of datasets required for transformer based models

`np` - Used in evaluation metrics for finding highest model output score

`accuracy_score` - Used to calculate overall accuracy of model

`precision_recall_fscore_support` - Used to calculate precision, recall, F1 and support of model

`AutoModelForSequenceClassification` - Used to load tansformer model for classification tasks

`DataCollatorWithPadding` - Used for padding training batches

`TrainingArguments` - Sets training arguments for model during training

`` - 

`` - 

`` - 

`` - 

`` - 



In [29]:
from datasets import load_dataset, DatasetDict
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Defining which Model

In [22]:
# Defining which model.
model_name = "distilbert-base-uncased"

# Dataset

`dataframe_check` function:

- Takes:
  - `dataframe`: a dataFrame containing training, validation and testing data


This function performs a couple of manual checks on the dataset for inspection by the user. For example, printing the first few values and checking the balance

In [4]:
def dataframe_check(dataframe):
    # Check first couple values
    print("Dataframe head:")
    print(dataframe.head())
    print("\n-----------------------------------")
    print("\nBalance of dataset")
    print(dataframe["label"].value_counts())
    print(dataframe["label"].value_counts(normalize=True))

Load the sst2 dataset.

In [5]:
sst2 = load_dataset("glue", "sst2")

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Convert to a dataframe and call the `dataframe_check` function to print some infomation about the training data

In [13]:
train_df = pd.DataFrame(sst2["train"])
dataframe_check(train_df)

Dataframe head:
                                            sentence  label  idx
0       hide new secretions from the parental units       0    0
1               contains no wit , only labored gags       0    1
2  that loves its characters and communicates som...      1    2
3  remains utterly satisfied to remain the same t...      0    3
4  on the worst revenge-of-the-nerds clichés the ...      0    4

-----------------------------------

Balance of dataset
label
1    37569
0    29780
Name: count, dtype: int64
label
1    0.557826
0    0.442174
Name: proportion, dtype: float64


# Create datset splits

The SST-2 has hidden labels for test data so only training and validation data will be taken into account. Training data will be split into training and testing.

In [14]:
train_test_split = sst2["train"].train_test_split(
    test_size = 0.2, # 20% used for testing
    seed = 42, # Seed for database split
    stratify_by_column = "label" # Keep label balance after split
)

# Remake DatasetDict with new split, since original had no test data labels
sst2_split = DatasetDict({
    "train": train_test_split["train"],
    "validation": sst2["validation"],
    "test": train_test_split["test"]
})

# Tokenisation of data

In [16]:
# Create the tokenizer for DistilBERT
tokenizer = AutoTokenizer.from_pretrained(model_name)

`tokenize` function:

- Takes:
  - `batch`: dictionary batch of rows from dataset

- Returns:
  - `tokenizer`: BertTokenizer containing numerical token IDs


Takes a batch of dataset rows and converts the sentences into token IDs. Also removed sentences longer than 128 tokens.

In [17]:
def tokenize(batch):
    return tokenizer(
        batch["sentence"], # select sentence text from dataset
        truncation=True, # cuts off text if it is too long
        max_length=128 # limits each sentence to 128 tokens. enough for SST2 due to short sentences
    )

In [18]:
# Applies tokenize function to each sentence
tokenized_sst2 = sst2_split.map(tokenize, batched=True)

# Remove columns no longer needed for training such as sentences, index.
tokenized_sst2 = tokenized_sst2.remove_columns(["sentence", "idx"])
# Rename column label to labels for compatibility
tokenized_sst2 = tokenized_sst2.rename_column("label", "labels")
# Return Pytorch tensor for tokenized_sst2
tokenized_sst2.set_format("torch")

# Setup Evaluation Metrics

`evaluation_metrics` function:

- Takes:
  - `evaluation_prediction`: Models predictions and correct labels

- Returns:
  - `Dictionary`: Dictionary of evaluation results including precision, recall, f1 and accuracy


This function takes models predictions and correct labels and returns a series of evaluation metrics about those predictions including accuracy, recall, f1 and precision

In [19]:
def evaluation_metrics(evaluation_prediciton):
    # Split incoming prediction into model prediction and correct labels
    raw_data, labels = evaluation_predictions

    # Convert output to class prediciton
    predictions = np.argmax(raw_data, axis=-1)

    # Compare predictions against labels
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, 
        predictions,
        average="binary" # binary as sst2 has binary labels 1 or 0.
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f2,
        "accuracy": accuracy
    }

# Loading Model, Data Collator and Training Arguments

In [24]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2 # Number of outputs (1, 0)
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

https://huggingface.co/docs/transformers/v4.15.0/en/main_classes/trainer#transformers.TrainingArguments

In [31]:
training_args = TrainingArguments(
    output_dir="./distilbert_sst2_baseline", # Output for trianing files
    eval_strategy="epoch", # Evaluation after each epoch
    save_strategy="epoch", # Save Model after each epoch
    learning_rate=2e-5, 
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True, # Keeps best saved model
    metric_for_best_model="f1", # How it decides which model is best
    greater_is_better=True, # Dictates higher F1 is better
    report_to="none" # Stop hugging face logging data
)